CHATBOT EVALUATION

In [25]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["LANGSMITH_API_KEY"] = os.getenv("LANGSMITH_API_KEY")
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["LANGSMITH_TRACING"] = "true"
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

creating the datapoints
- for a particular input -> what will be the output

In [26]:
from langsmith import Client
client = Client()

# defining dataset --> test data
dataset_name = "Chatbots Evaluation"
dataset = client.create_dataset(dataset_name)

client.create_examples(
    dataset_id=dataset.id,
    examples=[
        {
            "inputs": {"question": "What is LangChain ?"},
            "outputs": {"answer": "A framework for building LLM applications"}
        },
        {
            "inputs": {"question": "What is LangSmith?"},
            "outputs": {"answer": "A platform for observing and evaluating LLM applications"},
        },
        {
            "inputs": {"question": "What is OpenAI?"},
            "outputs": {"answer": "A company that creates Large Language Models"},
        },
        {
            "inputs": {"question": "What is Google?"},
            "outputs": {"answer": "A technology company known for search"},
        },
        {
            "inputs": {"question": "What is Mistral?"},
            "outputs": {"answer": "A company that creates Large Language Models"},
        }
    ]
)

{'example_ids': ['3b14bf57-fa7c-4145-ab36-e6c317cfc64b',
  'e69ed339-a9d6-486c-86ae-b97404458fe8',
  '4ef3550b-7b93-4d8c-9e77-4a3b8ff086b2',
  '2011fd9a-33cc-4e80-b556-78587fe6f066',
  'ab3b1076-d969-4560-b819-6792c0c5c8bb'],
 'count': 5,
 'as_of': '2026-08-01T12:46:28.020433659Z'}

metrics for eval - llm as judge

- correctness - metric

In [32]:
from openai import OpenAI
from langsmith import wrappers
groq_client = wrappers.wrap_openai(
    OpenAI(
        api_key=GROQ_API_KEY,
        base_url="https://api.groq.com/openai/v1"
    )
)
instructions = "You are an expert professor specialized in grading students' answers to questions."

def correctness(inputs:dict, outputs:dict, reference_outputs:dict) -> bool:
    user_content = f"""You are grading the following question:
    {inputs['question']}
    Here is the real answer:
    {reference_outputs['answer']}
    You are grading the following predicted answer:
    {outputs['response']}
    Respond with CORRECT or INCORRECT:
    Grade:
    """
    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        temperature=0,
        messages = [
            {"role":"system","content":instructions},
            {"role":"user","content":user_content}
        ]
    ).choices[0].message.content

    return response == "CORRECT"

- concision - checks whether actual output is less than 2x the length of expected result

In [33]:
def concision(outputs: dict, reference_outputs: dict) -> bool:
    return int(len(outputs["response"]) < 2 * len(reference_outputs["answer"]))

Running Evaluations

In [34]:
default_instructions = "Respond to the users question in a short, concise manner (one short sentence)."
# main chatbot function -> needs to be called for every input
def my_app(question: str,model: str="llama-3.3-70b-versatile",instructions: str = default_instructions) -> str:
    return groq_client.chat.completions.create(
        model=model,
        temperature=0,
        messages=[
            {"role":"system","content":instructions},
            {"role":"user","content":question}
        ]
    ).choices[0].message.content

In [43]:
from typing import Dict, Any
def ls_target(inputs: Dict[str, Any]) -> Dict[str, str]:
    return {
        "response": my_app(inputs["question"])
    }

In [37]:
experiment_results = client.evaluate(
    ls_target,
    data=dataset_name,
    evaluators=[correctness,concision],
    experiment_prefix="llama-3.3-70b-versatile"
)

View the evaluation results for experiment: 'llama-3.3-70b-versatile-0326b9a9' at:
https://smith.langchain.com/o/4a7e1065-23d2-4ab2-8bb4-13a5400bfe0e/datasets/7ee2c440-12da-4f4f-a8b7-a79dafa34c50/compare?selectedSessions=69570287-a072-4905-811c-fc8cb2cd3861




0it [00:00, ?it/s]

another experiment using diff model

In [44]:
from typing import Dict, Any
def ls_target1(inputs: Dict[str, Any]) -> Dict[str, str]:
    return {
        "response": my_app(inputs["question"],model="llama-3.1-8b-instant")
    }

In [45]:
experiment_results = client.evaluate(
    ls_target1,
    data=dataset_name,
    evaluators=[correctness,concision],
    experiment_prefix="llama-3.1-8b-instant"
)

View the evaluation results for experiment: 'llama-3.1-8b-instant-7c465f07' at:
https://smith.langchain.com/o/4a7e1065-23d2-4ab2-8bb4-13a5400bfe0e/datasets/7ee2c440-12da-4f4f-a8b7-a79dafa34c50/compare?selectedSessions=434fe53e-eb81-4d58-b33c-aaab15781ff0




0it [00:00, ?it/s]

Evaluation for RAG

In [2]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [3]:
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
embedding_model

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [6]:
urls = [
    "https://lilianweng.github.io/posts/2023-06-23-agent/",
    "https://lilianweng.github.io/posts/2023-03-15-prompt-engineering/",
    "https://lilianweng.github.io/posts/2023-10-25-adv-attack-llm/"
]

docs = [WebBaseLoader(url).load() for url in urls]
docs_list = [item for sublist in docs for item in sublist]

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=250,chunk_overlap=0
)

doc_splits = text_splitter.split_documents(docs_list)

vectorstore = InMemoryVectorStore.from_documents(
    documents=doc_splits,
    embedding=embedding_model
)

retriever = vectorstore.as_retriever(k=6)

In [7]:
retriever

VectorStoreRetriever(tags=['InMemoryVectorStore', 'HuggingFaceEmbeddings'], vectorstore=<langchain_core.vectorstores.in_memory.InMemoryVectorStore object at 0x00000293042C4230>, search_kwargs={})

In [8]:
retriever.invoke("What is agents?")

[Document(id='865f6774-6fd0-4df9-b765-0e9b510149f6', metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/', 'title': "LLM Powered Autonomous Agents | Lil'Log", 'description': 'Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.\nAgent System Overview\nIn a LLM-powered autonomous agent system, LLM functions as the agent’s brain, complemented by several key components:\n\nPlanning\n\nSubgoal and decomposition: The agent breaks down large tasks into smaller, manageable subgoals, enabling efficient handling of complex tasks.\nReflection and refinement: The agent can do self-criticism and self-reflection over past actions, learn from mistakes and refine them for future steps, 

In [12]:
from langchain.chat_models import init_chat_model
llm = init_chat_model(model="groq:llama-3.3-70b-versatile")
llm

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}}, output_version=None, profile={'name': 'Llama 3.3 70B Versatile', 'release_date': '2024-12-06', 'last_updated': '2024-12-06', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x00000293086E5B50>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000293071D79B0>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [13]:
from langsmith import traceable

@traceable
def rag_bot(question: str) -> dict:
    docs = retriever.invoke(question)
    docs_string = "".join(doc.page_content for doc in docs)
    instructions = f"""You are a helpful assistant who is good at analyzing source information and answering questions.
    Documents:
    {docs_string}
    """
    ai_msg = llm.invoke([
        {"role":"system","content":instructions},
        {"role":"user","content":question}
    ])
    return {"answer":ai_msg.content,"documents":docs}

In [14]:
rag_bot("What is agents")

{'answer': 'In the context of the provided text, an agent refers to a type of autonomous system that uses a Large Language Model (LLM) as its core controller. This agent is designed to perform tasks, make decisions, and interact with its environment in a way that is similar to human-like intelligence.\n\nIn general, an agent can be defined as a system that:\n\n1. Perceives its environment through sensors or observations\n2. Has a set of goals or objectives to achieve\n3. Can take actions to affect its environment\n4. Can learn from its experiences and adapt to changing situations\n\nIn the case of LLM-powered autonomous agents, the LLM serves as the agent\'s "brain," enabling it to reason, plan, and make decisions based on the information it receives from its environment.\n\nSome key characteristics of agents include:\n\n1. **Autonomy**: Agents can operate independently, making decisions and taking actions without human intervention.\n2. **Reactivity**: Agents can respond to changes in

Building Test Data for Experiments

In [16]:
from langsmith import Client

client = Client()
examples = [
    {
        "inputs": {"question": "How does the ReAct agent use self-reflection? "},
        "outputs": {"answer": "ReAct integrates reasoning and acting, performing actions - such tools like Wikipedia search API - and then observing / reasoning about the tool outputs."},
    },
    {
        "inputs": {"question": "What are the types of biases that can arise with few-shot prompting?"},
        "outputs": {"answer": "The biases that can arise with few-shot prompting include (1) Majority label bias, (2) Recency bias, and (3) Common token bias."},
    },
    {
        "inputs": {"question": "What are five types of adversarial attacks?"},
        "outputs": {"answer": "Five types of adversarial attacks are (1) Token manipulation, (2) Gradient based attack, (3) Jailbreak prompting, (4) Human red-teaming, (5) Model red-teaming."},
    }
]

dataset_name = "RAG Test Evaluation"
dataset = client.create_dataset(dataset_name=dataset_name)
client.create_examples(
    dataset_id=dataset.id,
    examples=examples
)

{'example_ids': ['044d9fa1-2696-4496-ada5-82bfaed0243a',
  'b7f0aecb-fcf4-4952-87f8-0e80fca98711',
  '06518fe8-cb50-4914-a5c1-0ddca46cd042'],
 'count': 3,
 'as_of': '2026-08-01T14:21:11.477817197Z'}

# Evaluators

LLM as the Judge

1) Correctness: Response vs Reference Answer

In [ ]:
from typing_extensions import Annotated,TypedDict

#correctness output schema
class CorrectnessGrade(TypedDict):
    explanation: Annotated[str,...,"Explain your reasoning for the score"]
    correct: Annotated[bool,...,"True if the answer is correct, False Otherwise"]

#correctness prompt
correctness_instructions = """You are a teacher grading a quiz. 
You will be given a QUESTION, the GROUND TRUTH (correct) ANSWER, and the STUDENT ANSWER. 
Here is the grade criteria to follow:
(1) Grade the student answers based ONLY on their factual accuracy relative to the ground truth answer. 
(2) Ensure that the student answer does not contain any conflicting statements.
(3) It is OK if the student answer contains more information than the ground truth answer, as long as it is factually accurate relative to the  ground truth answer.
Correctness:
A correctness value of True means that the student's answer meets all of the criteria.
A correctness value of False means that the student's answer does not meet all of the criteria.
Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 
Avoid simply stating the correct answer at the outset."""

from langchain_groq import ChatGroq

grader_llm = ChatGroq(model="llama-3.3-70b-versatile",temperature=0).with_structured_output(CorrectnessGrade,method="json_schema",strict=True)


In [20]:
def correctness(inputs: dict,outputs: dict, reference_outputs: dict) -> bool:
    """An evaluator for RAG answer accuracy"""
    answers = f"""
    QUESTION: {inputs['question']}
    GROUND TRUTH ANSWER: {reference_outputs["answer"]}
    STUDENT ANSWER: {outputs["answer"]}"""

    grade = grader_llm.invoke([
        {"role":"system","content": correctness_instructions},
        {"role":"user","content":answers}
    ])

    return grade["correct"]

2) Answer Relevance: Is generated answer relevant to the question

In [ ]:
# relevance output schema
class RelevanceGrade(TypedDict):
    explanation: Annotated[str,...,"Explain your reasoning for the score"]
    relevant: Annotated[bool,...,"Provide the score on wether the answer addresses the question"]

# relevance prompt
relevance_instructions="""You are a teacher grading a quiz. 
You will be given a QUESTION and a STUDENT ANSWER. 
Here is the grade criteria to follow:
(1) Ensure the STUDENT ANSWER is concise and relevant to the QUESTION
(2) Ensure the STUDENT ANSWER helps to answer the QUESTION
Relevance:
A relevance value of True means that the student's answer meets all of the criteria.
A relevance value of False means that the student's answer does not meet all of the criteria.
Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 
Avoid simply stating the correct answer at the outset."""


relevance_llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0).with_structured_output(RelevanceGrade, method="json_schema", strict=True)

In [23]:
def relevance(inputs: dict,outputs: dict) -> bool:
    """A simple evaluator for RAG answer helpfulness"""
    answer = f"QUESTION: {inputs['question']}\n STUDENT ANSWER: {outputs['answer']}"
    grade = relevance_llm.invoke([
        {"role":"system","content":relevance_instructions},
        {"role":"user","content":answer}
    ])
    return grade['relevant']

3) Groundedness: Response vs Retrieved Docs

does generated answer contain the grounded truth / info from retrieved docs

In [24]:
# groundedness output schema
class GroundedGrade(TypedDict):
    explanation: Annotated[str,...,"Explain your reasoning for the score"]
    grounded: Annotated[bool,...,"Provide the score on if the answer hallucinates from the documents"]

# Groundedness prompt
grounded_instructions = """You are a teacher grading a quiz. 
You will be given FACTS and a STUDENT ANSWER. 
Here is the grade criteria to follow:
(1) Ensure the STUDENT ANSWER is grounded in the FACTS. 
(2) Ensure the STUDENT ANSWER does not contain "hallucinated" information outside the scope of the FACTS.
Grounded:
A grounded value of True means that the student's answer meets all of the criteria.
A grounded value of False means that the student's answer does not meet all of the criteria.
Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 
Avoid simply stating the correct answer at the outset."""

grounded_llm = ChatGroq(model="llama-3.3-70b-versatile",temperature=0).with_structured_output(GroundedGrade,method="json_schema",strict=True)

In [26]:
def groundedness(inputs: dict,outputs: dict) -> bool:
    """A simple evaluator for RAG answer groundness"""
    doc_string = "\n\n".join(doc.page_content for doc in outputs["documents"])
    answer = f"FACTS: {doc_string}\nSTUDENT ANSWER: {outputs['answer']}"
    grade = grounded_llm.invoke([
        {"role":"system","content":grounded_instructions},
        {"role":"user","content":answer}
    ])
    return grade["grounded"]

4) Retrieval Relevance: Retrieved Docs vs Input

are retrieved docs relevant to the input question

In [28]:
# RetrievalRelevance Output Schema
class RetrievalRelevanceGrade(TypedDict):
    explanation: Annotated[str,...,"Explain your reasoning for the score"]
    relevance: Annotated[bool,...,"True if the retrieved documents are relevant to the question, False otherwise"]

# Retrieval Relevance Prompt
retrieval_relevance_instructions = """You are a teacher grading a quiz. 
You will be given a QUESTION and a set of FACTS provided by the student. 
Here is the grade criteria to follow:
(1) You goal is to identify FACTS that are completely unrelated to the QUESTION
(2) If the facts contain ANY keywords or semantic meaning related to the question, consider them relevant
(3) It is OK if the facts have SOME information that is unrelated to the question as long as (2) is met
Relevance:
A relevance value of True means that the FACTS contain ANY keywords or semantic meaning related to the QUESTION and are therefore relevant.
A relevance value of False means that the FACTS are completely unrelated to the QUESTION.
Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 
Avoid simply stating the correct answer at the outset."""

retrieval_relevance_llm = ChatGroq(model="llama-3.3-70b-versatile",temperature=0).with_structured_output(RetrievalRelevanceGrade,method="json_schema",strict=True)

In [30]:
def retrieval_relevance(inputs: dict,outputs: dict) -> bool:
    """An evaluator for document relevance"""
    doc_string = "\n\n".join(doc.page_content for doc in outputs["documents"])
    answer = f"FACTS: {doc_string}\n QUESTION: {inputs['question']}"

    grade = retrieval_relevance_llm.invoke([
        {"role":"system","content":retrieval_relevance_instructions},
        {"role":"user","content":answer}
    ])
    return grade["relevant"]

Running the Evaluation

In [32]:
def target(inputs: dict) -> dict:
    return rag_bot(inputs["question"])

experiment_results = client.evaluate(
    target,
    data=dataset_name,
    evaluators=[correctness,groundedness,relevance,retrieval_relevance],
    experiment_prefix="rag-doc-relevance",
    metadata={"version":"LCEL context, llama-3.3-70b-versatile-preview"}
)

experiment_results.to_pandas()

View the evaluation results for experiment: 'rag-doc-relevance-d845c262' at:
https://smith.langchain.com/o/4a7e1065-23d2-4ab2-8bb4-13a5400bfe0e/datasets/94c1bc6e-87d1-4f30-b217-a8228f267a8e/compare?selectedSessions=20ab2d4d-edcb-4cb8-b70e-c6929a0f5e0a




0it [00:00, ?it/s]

Error running target function: Error code: 400 - {'error': {'message': 'This model does not support response format `json_schema`. See supported models at https://console.groq.com/docs/structured-outputs#supported-models', 'type': 'invalid_request_error', 'param': 'response_format'}}
Traceback (most recent call last):
  File "C:\Users\Dell\AppData\Roaming\Python\Python312\site-packages\langsmith\evaluation\_runner.py", line 1976, in _forward
    fn(*args, langsmith_extra=langsmith_extra)
  File "C:\Users\Dell\AppData\Roaming\Python\Python312\site-packages\langsmith\run_helpers.py", line 777, in wrapper
    function_result = run_container["context"].run(
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Dell\AppData\Local\Temp\ipykernel_3960\1343817939.py", line 2, in target
    return rag_bot(inputs["question"])
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Dell\AppData\Roaming\Python\Python312\site-packages\langsmith\run_helpers.py", line 777, in wrapper
 

,inputs.question,outputs.output,error,reference.answer,feedback.correctness,feedback.groundedness,feedback.relevance,feedback.retrieval_relevance,execution_time,example_id,id
0,How does the ReAct agent use self-reflection?,None,"BadRequestError(""Error code: 400 - {'error': {...","ReAct integrates reasoning and acting, perform...",None,None,None,None,0.162951,044d9fa1-2696-4496-ada5-82bfaed0243a,019fbde3-ce47-7c93-97d9-09c0bca64fd3
1,What are five types of adversarial attacks?,None,"BadRequestError(""Error code: 400 - {'error': {...",Five types of adversarial attacks are (1) Toke...,None,None,None,None,0.071367,06518fe8-cb50-4914-a5c1-0ddca46cd042,019fbde3-cf11-7350-9298-da6c9eef0428
2,What are the types of biases that can arise wi...,None,"BadRequestError(""Error code: 400 - {'error': {...",The biases that can arise with few-shot prompt...,None,None,None,None,0.069283,b7f0aecb-fcf4-4952-87f8-0e80fca98711,019fbde3-cf6b-7c41-bbe5-9e4fe24fd125
